# LTIMindtree Azure Data Engineer Interview Experience

**Position:** Data Engineer
**Experience:** 4+ Years

---

## ROUND 2

1. Azure ETL pipeline design
2. Incremental load strategies
3. Spark fault tolerance
4. Pipeline failure handling
5. ADF triggers types
6. ADF Integration Runtime
7. Cluster vs Client mode Spark
8. Spark performance tuning
9. SQL 2nd highest salary
10. CDC pipeline design
11. Partition vs Bucketing
12. Handling late arriving data

2. Incremental load strategies

## 1. Incremental Load Strategies in Azure ETL

**Incremental loading** means processing only **new or changed records** instead of loading the complete dataset every time.

```text
Full Load:
Source ───────────────→ Target
        ALL records

Incremental Load:
Source ──→ New/Changed Records ──→ Target
```

This reduces **processing time, data movement, compute cost, and load on source systems**.

### 1. Watermark / Last Modified Date

One of the most common approaches.

Suppose the source has:

```text
customer_id | name | modified_date
101         | John | 2026-08-08 10:15
102         | Mike | 2026-08-09 09:20
```

Maintain the last successfully processed timestamp:

```text
Last Watermark = 2026-08-08 23:59:59
```

Then query only changed records:

```sql
SELECT *
FROM Customer
WHERE modified_date > @LastWatermark
```

After successful processing, update the watermark.

```text
Control Table
-------------------------------
Table       LastWatermark
Customer    2026-08-09 09:20
Orders      2026-08-09 10:45
```

**ADF implementation:**

```text
Lookup Watermark
       ↓
Copy Incremental Data
       ↓
Databricks Transformation
       ↓
Update Watermark
```

---

### 2. High-Watermark Using ID

If the source has a continuously increasing ID:

```sql
SELECT *
FROM Orders
WHERE order_id > @LastProcessedID
```

For example:

```text
Last Processed ID = 10000

Next Run:
WHERE order_id > 10000
```

This is simple and efficient, but **doesn't handle updates to old records** unless another mechanism identifies updates.

---

### 3. Change Data Capture (CDC)

CDC tracks **INSERT, UPDATE, and DELETE** operations.

```text
Source DB
   ↓
CDC
   ↓
Changed Records
   ↓
ADF
   ↓
ADLS / Databricks
```

This is useful when you need to capture both new and modified records and, depending on the source CDC implementation, deletions.

---

### 4. Change Tracking

Database **Change Tracking** can identify which rows changed since a previous synchronization point.

It is generally lighter than full CDC but provides less detailed historical change information.

Typical pattern:

```text
Last Sync Version
       ↓
Get Changed Records
       ↓
Process Changes
       ↓
Store New Version
```

---

### 5. File-Based Incremental Load

For files arriving in ADLS/SFTP, you can process only files that haven't already been processed.

Maintain a metadata table:

```text
File Name          Status
customer_01.csv    Processed
customer_02.csv    Processed
customer_03.csv    Pending
```

ADF checks the metadata before processing.

```text
SFTP / ADLS
     ↓
Get New Files
     ↓
Check Metadata
     ↓
Process Unprocessed Files
     ↓
Update Metadata
```

---

### 6. MERGE-Based Incremental Load

After extracting incremental records, use `MERGE` to update the target.

For example, in Delta Lake:

```sql
MERGE INTO target t
USING source s
ON t.customer_id = s.customer_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
```

This is particularly useful for **upsert scenarios**.

---

## Which Strategy Should You Choose?

| Strategy        | Best For                               |
| --------------- | -------------------------------------- |
| Watermark       | Source has `modified_date`             |
| Increasing ID   | Insert-only data                       |
| CDC             | Need INSERT/UPDATE/DELETE              |
| Change Tracking | Need lightweight change detection      |
| File Metadata   | File-based ingestion                   |
| MERGE           | Applying incremental changes to target |

### ⭐ Interview Answer

> **"For incremental loading, I first understand how the source identifies new and changed records. If the source has a reliable last-modified timestamp, I prefer a watermark-based approach. I maintain the last successfully processed watermark in a control table and pass it as a parameter to the source query. For databases supporting CDC or Change Tracking, I can use those mechanisms when I need reliable change detection, including updates and deletes. For file-based sources, I maintain file-level metadata to identify already processed files. After ingestion, I use Delta MERGE or an equivalent upsert strategy to apply changes to the target. I also update the watermark only after successful processing to make the pipeline restartable and avoid data loss."**

### 🔥 Important Interview Point

**Never update the watermark before the data is successfully processed.**

Correct:

```text
Read Watermark
      ↓
Extract Data
      ↓
Transform
      ↓
Load Successfully
      ↓
Update Watermark ✅
```

If you update it before the load succeeds, a pipeline failure can cause records to be **skipped in the next run**.



3. Spark fault tolerance

## Spark Fault Tolerance

**Spark fault tolerance** means Spark can recover from failures of executors or tasks without restarting the entire application.

The main mechanism is **RDD/DataFrame lineage**.

### How it works

```text
Driver
  |
  +---- Executor 1
  |       ├── Task A
  |       └── Task B
  |
  +---- Executor 2
          ├── Task C
          └── Task D
```

Suppose **Executor 2 fails**:

```text
Executor 2 ❌
   |
Task C/D lost
   ↓
Spark identifies lost partitions
   ↓
Recomputes them from lineage
   ↓
Runs tasks on another executor
   ↓
Job continues
```

### 1. Lineage

Spark keeps track of **how a dataset was created through transformations**.

For example:

```python
df1 = spark.read.parquet("/data")
df2 = df1.filter("salary > 50000")
df3 = df2.select("name", "salary")
```

Spark knows the dependency:

```text
Source
  ↓
filter()
  ↓
select()
```

If a partition of `df3` is lost, Spark can recompute the required data from the previous stages.

### 2. DAG and Task Re-execution

Spark creates a **DAG (Directed Acyclic Graph)** of operations.

If a task fails because an executor crashes or encounters a transient issue, Spark can **retry the task**.

```text
Task 1 → Failed
          ↓
       Retry
          ↓
       Success
```

If repeated attempts fail, Spark eventually marks the stage/job as failed.

### 3. Shuffle Fault Tolerance

Shuffle data is distributed across executors.

If shuffle data is lost, Spark can often **recompute the missing shuffle data** by rerunning the relevant upstream tasks.

This is why shuffle operations such as:

```python
df.groupBy("department").count()
```

can recover from executor failures.

### 4. Cache and Persistence

Caching improves performance but doesn't necessarily mean the cached data is permanently safe.

```python
df.cache()
```

If cached partitions are lost, Spark can generally **recompute them from lineage**.

With:

```python
df.persist()
```

you can choose different storage levels depending on your requirements.

### 5. Checkpointing

For very long or complex lineage chains, **checkpointing** can truncate the lineage.

```python
df.checkpoint()
```

Conceptually:

```text
Long Lineage
A → B → C → D → E → F
              ↓
         Checkpoint
              ↓
              F
```

If something fails after the checkpoint, Spark doesn't need to recompute the entire lineage from `A`.

---

## Spark Fault Tolerance vs Traditional Systems

| Feature          | Spark                               |
| ---------------- | ----------------------------------- |
| Task failure     | Task can be retried                 |
| Executor failure | Lost partitions can be recomputed   |
| Data recovery    | Lineage                             |
| Shuffle recovery | Recompute upstream data when needed |
| Long lineage     | Checkpointing                       |
| Cached data lost | Recompute from lineage              |

### ⭐ Interview Answer

> **"Spark provides fault tolerance mainly through lineage. Instead of replicating every intermediate dataset, Spark remembers the transformations used to create the data. If an executor or partition is lost, Spark can recompute the missing partition from its lineage. Spark also retries failed tasks and can recompute lost shuffle data. For applications with very long lineage, we can use checkpointing to persist the state and truncate the lineage. So the key concepts behind Spark fault tolerance are lineage, task retry, recomputation, and checkpointing."**

### 🔥 One-line answer

**"Spark achieves fault tolerance primarily through RDD/DataFrame lineage, allowing lost partitions to be recomputed and failed tasks to be retried rather than restarting the entire job."**


In [0]:
# 4. Pipeline failure handling
# For pipeline failure handling, I first categorize failures as transient or permanent. For transient failures such as network timeouts or throttling, I configure retries with an appropriate retry interval. For permanent failures, I use ADF failure dependencies to route execution to an error-handling process. I capture the pipeline run ID, activity name, error message, timestamps and other metadata in an audit table. I then send alerts through the monitoring/notification layer. I also implement data-quality checks and make the pipeline idempotent and restartable so that a failed rerun doesn't create duplicate or inconsistent data.

5. ADF triggers types

## ADF Trigger Types

In **Azure Data Factory (ADF)**, a **trigger** defines **when and under what condition a pipeline should start**.

There are **4 commonly discussed trigger types**:

### 1. Schedule Trigger

Runs a pipeline at a **specific time or recurring schedule**.

**Example:** Run an ETL pipeline every day at 2 AM.

```text
Schedule Trigger
      ↓
ADF Pipeline
      ↓
ETL Processing
```

Typical use cases:

* Daily batch processing
* Weekly/monthly reports
* Periodic data loads

---

### 2. Tumbling Window Trigger

Runs a pipeline for **fixed, continuous time windows**.

Example:

```text
00:00 ─── 01:00
01:00 ─── 02:00
02:00 ─── 03:00
```

Each window is processed independently and has a defined start/end time.

It is useful when you need:

* Time-based incremental processing
* Dependency between windows
* Backfilling historical windows
* Reliable batch processing

**Interview point:** Unlike a simple schedule trigger, tumbling windows maintain **state for each time window** and support dependencies between windows.

---

### 3. Event-Based Trigger

Starts a pipeline when a specific **event occurs**.

A common example is a new file arriving in Azure Storage.

```text
File Uploaded to ADLS
        ↓
Storage Event
        ↓
ADF Event Trigger
        ↓
Pipeline
```

Example:

```text
customer_20260810.csv
        ↓
Uploaded
        ↓
Trigger fires
        ↓
Process file
```

Useful for **near-real-time file ingestion**.

---

### 4. Manual / On-Demand Trigger

The pipeline is started manually by a developer, operator, REST API, or another process.

```text
User / API
    ↓
ADF Pipeline
```

Useful for:

* Testing
* Debugging
* Ad-hoc processing
* Backfilling data

---

## Quick Comparison

| Trigger              | Starts When              | Example                 |
| -------------------- | ------------------------ | ----------------------- |
| **Schedule**         | Specific time/recurrence | Every day at 2 AM       |
| **Tumbling Window**  | Fixed time window        | Process hourly data     |
| **Event-Based**      | Event occurs             | New file arrives        |
| **Manual/On-Demand** | Explicit invocation      | Developer runs pipeline |



**Easy way to remember:**
**Schedule = Time | Tumbling = Time Window | Event = Event | Manual = User/API**.


6. ADF Integration Runtime

## ADF Integration Runtime (IR)

**Integration Runtime (IR)** is the **compute infrastructure used by Azure Data Factory to connect, move, and process data between different environments**.

Think of IR as the **bridge between ADF and your data sources/targets**.

```text
Source
   ↓
Integration Runtime
   ↓
Azure Data Factory Pipeline
   ↓
Target
```

### Types of Integration Runtime

ADF primarily has **3 types of Integration Runtime**:

### 1. Azure Integration Runtime

This is the most commonly used option.

It is **managed by Azure** and is used for cloud-based data movement and activities.

```text
Azure SQL
    ↓
Azure IR
    ↓
ADLS Gen2
```

Typical use cases:

* Azure SQL → ADLS
* ADLS → Azure SQL
* Cloud-to-cloud data movement
* Data flows
* Connecting to supported cloud services

You don't have to manage the underlying infrastructure.

---

### 2. Self-Hosted Integration Runtime (SHIR)

**Self-hosted IR** is installed on a machine that you manage.

It is mainly used when ADF needs to access **on-premises or network-restricted data sources**.

```text
On-Premises SQL Server
        ↓
Self-Hosted IR
        ↓
Azure Data Factory
        ↓
ADLS
```

For example:

```text
On-Prem Oracle
      ↓
   SHIR
      ↓
    ADF
      ↓
   ADLS
```

You are responsible for:

* Installing SHIR
* Maintaining the server/VM
* Network connectivity
* Availability
* Upgrades

---

### 3. Azure-SSIS Integration Runtime

This is specifically designed to **lift and shift existing SQL Server Integration Services (SSIS) packages to Azure**.

```text
Existing SSIS Packages
        ↓
Azure-SSIS IR
        ↓
ADF
```

It's useful when an organization has significant existing SSIS workloads and wants to migrate them to Azure without completely rewriting them.

---

## Azure IR vs Self-Hosted IR

| Feature                    | Azure IR              | Self-Hosted IR    |
| -------------------------- | --------------------- | ----------------- |
| Infrastructure             | Azure managed         | Customer managed  |
| On-premises access         | Limited/not direct    | ✅ Yes             |
| Cloud data movement        | ✅ Yes                 | ✅ Yes             |
| Maintenance                | Low                   | Higher            |
| Network-restricted sources | Depends on networking | ✅ Common use case |
| Scaling                    | Azure managed         | You manage nodes  |

### ⭐ Interview Answer

> **"Integration Runtime is the compute infrastructure used by Azure Data Factory to perform data movement and execute certain activities. ADF mainly provides Azure Integration Runtime, Self-Hosted Integration Runtime, and Azure-SSIS Integration Runtime. Azure IR is generally used for cloud-to-cloud data movement and is managed by Azure. Self-Hosted IR is installed and managed by us and is commonly used to connect ADF with on-premises or network-restricted data sources. Azure-SSIS IR is used to run existing SSIS packages in Azure."**

### 🔥 Important Interview Question

**Q: Why do we need Self-Hosted IR if ADF is a cloud service?**

**Answer:**

> **"Because ADF itself cannot directly access many on-premises or private network data sources. Self-Hosted IR acts as the secure bridge between the private network and ADF, allowing data movement without exposing the source directly to the public internet."**

**Easy way to remember:**

**Azure IR → Cloud**
**Self-Hosted IR → On-Prem/Private Network**
**Azure-SSIS IR → SSIS Migration**


7. Cluster vs Client mode Spark

## Cluster Mode vs Client Mode in Spark

The main difference is **where the Spark Driver runs**.

### 1. Client Mode

In **client mode**, the **Spark Driver runs on the machine where you submit the Spark application**, while executors run on the cluster.

```text
Your Machine / Edge Node
        |
      Driver
        |
   Spark Cluster
   ┌────┴────┐
Executor 1  Executor 2
```

**Example:**

```bash
spark-submit \
  --master yarn \
  --deploy-mode client \
  app.py
```

**Use case:** Interactive development, debugging, or when you want the driver close to the user/application.

**Important:** If the client machine goes down, the Spark application can fail because the Driver is running there.

---

### 2. Cluster Mode

In **cluster mode**, Spark launches the **Driver inside the cluster**.

```text
Your Machine
     |
 spark-submit
     |
     v
Spark Cluster
     |
   Driver
   /    \
Executor  Executor
```

**Example:**

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  app.py
```

Your machine only submits the application; the actual Driver runs inside the cluster.

**Use case:** Production jobs, scheduled pipelines, and long-running applications.

---

## Key Difference

| Feature                              | Client Mode                | Cluster Mode                             |
| ------------------------------------ | -------------------------- | ---------------------------------------- |
| Driver location                      | Client machine / edge node | Cluster                                  |
| Executors                            | Cluster                    | Cluster                                  |
| Good for                             | Development / interactive  | Production                               |
| Client connection required           | Generally yes              | Mainly for submission                    |
| Driver failure if client disconnects | Possible                   | Not dependent on client after submission |
| Production preference                | Less common                | ✅ Preferred                              |

### ⭐ Interview Answer

> **"The key difference between Spark client and cluster mode is the location of the Driver. In client mode, the Driver runs on the machine from which the application is submitted, while executors run on the cluster. In cluster mode, both the Driver and Executors run inside the cluster. I generally prefer cluster mode for production workloads because the application is self-contained within the cluster and isn't dependent on the client machine after submission."**

### 🔥 Easy Way to Remember

**Client Mode:**
`Driver → Outside cluster`

**Cluster Mode:**
`Driver → Inside cluster`

**One important clarification:** Client vs Cluster mode is specifically a **Spark deployment mode**. The exact behavior can vary slightly depending on the cluster manager, such as **YARN, Kubernetes, or standalone**.


8. Spark performance tuning

## Spark Performance Tuning

**Spark performance tuning** is about reducing **execution time, memory usage, shuffle, and compute cost** while processing large datasets.

For an interview, I recommend explaining it across **data, Spark code, joins, partitions, memory, and cluster configuration**.

### 1. Optimize Data Partitioning

Avoid having too many or too few partitions.

```python
df = df.repartition("customer_id")
```

For reducing partitions:

```python
df = df.coalesce(10)
```

Good partitioning helps distribute work evenly across executors.

**Watch out for:** data skew, where one partition contains significantly more data than others.

---

### 2. Reduce Shuffle

**Shuffle is one of the biggest performance bottlenecks in Spark.**

Operations that commonly cause shuffle:

```python
groupBy()
join()
distinct()
orderBy()
repartition()
```

For example:

```python
df.groupBy("department").count()
```

creates a shuffle because data needs to be moved across executors.

Try to minimize unnecessary shuffles and avoid repeated repartitioning.

---

### 3. Optimize Joins

For a small lookup table, use a **Broadcast Join**.

```python
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "customer_id"
)
```

Instead of shuffling both datasets, Spark broadcasts the small dataset to executors.

```text
Large Dataset ─────────┐
                       ├── Join
Small Dataset → Broadcast
```

---

### 4. Handle Data Skew

If one key has much more data than others, one task may take significantly longer.

Example:

```text
Partition 1 → 10 GB
Partition 2 → 100 MB
Partition 3 → 120 MB
Partition 4 → 90 MB
```

Partition 1 becomes the **straggler**.

Possible solutions:

* Salting
* Broadcast join
* AQE skew join optimization
* Better partitioning
* Filtering unnecessary data

---

### 5. Use Predicate Pushdown

Filter data as early as possible.

Instead of:

```python
df = spark.read.parquet(path)
df = df.filter("salary > 50000")
```

Spark can push the filter down toward the data source when supported.

For large datasets, this can significantly reduce the amount of data read.

---

### 6. Select Only Required Columns

Avoid:

```python
df.select("*")
```

when you only need a few columns.

Prefer:

```python
df.select("customer_id", "name", "salary")
```

This reduces:

* I/O
* Memory usage
* Network transfer
* Serialization overhead

---

### 7. Cache/Persist Carefully

If the same DataFrame is used multiple times:

```python
df.cache()
```

or:

```python
df.persist()
```

Example:

```python
df = expensive_transformation()

df.cache()

df.count()
df.groupBy("department").count()
```

But **don't cache everything**.

Caching consumes executor memory and can actually hurt performance if used unnecessarily.

---

### 8. Use Efficient File Formats

Prefer:

```text
Parquet
Delta Lake
```

over formats such as CSV for large-scale processing.

Columnar formats provide:

* Column pruning
* Predicate pushdown
* Compression
* Efficient storage

---

### 9. Avoid Small Files

Thousands of tiny files can create excessive task scheduling and metadata overhead.

Instead of:

```text
10,000 × 1 MB files
```

prefer appropriately sized files.

For Delta tables, techniques such as compaction/optimization can help manage file sizes.

---

### 10. Adaptive Query Execution (AQE)

AQE allows Spark to optimize the query **at runtime** based on actual statistics.

It can help with:

* Coalescing shuffle partitions
* Handling skewed joins
* Optimizing join strategies

For example:

```python
spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
)
```

Modern Spark environments often have AQE enabled by default.

---

### 11. Tune Shuffle Partitions

Spark's shuffle partition configuration affects operations such as joins and aggregations.

```python
spark.conf.set(
    "spark.sql.shuffle.partitions",
    "200"
)
```

Too few partitions:

```text
Large partitions → Slow tasks / memory pressure
```

Too many:

```text
Tiny tasks → Scheduling overhead
```

The right value depends on **data volume and cluster resources**.

---

### 12. Avoid Collecting Large Data

Avoid:

```python
df.collect()
```

for large datasets.

`collect()` brings **all records to the Driver**, potentially causing:

```text
Driver
  ↓
OutOfMemoryError ❌
```

Use alternatives such as:

```python
df.show()
df.limit(100).collect()
```

when you only need a small sample.

---

## Practical Performance Tuning Flow

```text
Spark Job is Slow
       ↓
Check Spark UI
       ↓
Identify Slow Stage
       ↓
Check Shuffle
       ↓
Check Data Skew
       ↓
Check Partition Count
       ↓
Check Join Strategy
       ↓
Check Memory / GC
       ↓
Optimize Code & Data
       ↓
Re-run & Compare
```

### What I Check in Spark UI

When debugging a slow Spark job, I typically look at:

* **Stages**
* **Tasks**
* Shuffle read/write
* Task duration
* Input/output size
* Spill to disk
* Executor memory
* GC time
* Failed tasks
* Number of partitions

---

## ⭐ Strong Interview Answer

> **"For Spark performance tuning, I first identify the bottleneck using Spark UI rather than blindly changing configurations. I look at stages, shuffle read/write, task distribution, spills, GC time and data skew. Then I optimize the data by filtering early, selecting only required columns, using efficient formats such as Parquet or Delta, and managing partition sizes. For joins, I use broadcast joins when one dataset is sufficiently small and address skew using techniques such as salting or AQE. I also avoid unnecessary shuffles, cache only reused datasets, tune shuffle partitions when required, and avoid collecting large datasets to the Driver. Finally, I compare the execution metrics before and after optimization."**

### 🔥 Easy Framework to Remember

**F → P → S → J → M → C**

* **F** — Filter early
* **P** — Partition properly
* **S** — Reduce Shuffle
* **J** — Optimize Joins
* **M** — Manage Memory
* **C** — Cache carefully



In [0]:
# 9. SQL 2nd highest salary

SELECT salary
FROM (
    SELECT salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM Employee
) t
WHERE rnk = 2;

10. CDC pipeline design

## CDC Pipeline Design — Azure Data Engineering

**CDC (Change Data Capture)** is used to capture only the **INSERT, UPDATE, and DELETE** changes from a source system and propagate those changes to the target instead of performing a full load every time.

A production-ready Azure CDC architecture can look like this:

```text
             Source Database
          ┌───────────────────┐
          │ SQL / Oracle / SAP │
          └─────────┬─────────┘
                    │
                    │ CDC
                    ▼
          ┌───────────────────┐
          │ Azure Data Factory │
          │   Orchestration    │
          └─────────┬─────────┘
                    │
                    ▼
          ┌───────────────────┐
          │    ADLS Gen2      │
          │  Bronze / Raw CDC │
          └─────────┬─────────┘
                    │
                    ▼
          ┌───────────────────┐
          │    Databricks     │
          │     PySpark       │
          └─────────┬─────────┘
                    │
                    ▼
          ┌───────────────────┐
          │ Delta Silver/Gold │
          │   MERGE / UPSERT  │
          └─────────┬─────────┘
                    │
                    ▼
                 Power BI
```

### 1. Capture Changes

The source database maintains information about changed records.

For example:

```text
customer_id | name  | operation
------------|-------|----------
101         | John  | INSERT
102         | Mike  | UPDATE
103         | NULL  | DELETE
```

Depending on the source, CDC can be implemented using:

* Database CDC
* Change Tracking
* Transaction logs
* SCN/LSN
* Timestamp/watermark
* Source-specific CDC mechanisms

---

### 2. ADF Orchestration

**Azure Data Factory** can orchestrate the CDC process.

Typical pipeline:

```text
Get Last CDC Position
        ↓
Read Changed Records
        ↓
Copy to ADLS
        ↓
Trigger Databricks
        ↓
Validate
        ↓
Update CDC Position
```

Maintain a **control table**:

```text
table_name | last_cdc_position
-----------|-------------------
customer   | 987654
orders     | 456789
```

The important point is that the CDC position should be updated **only after successful processing**.

---

### 3. Bronze Layer

Store the CDC events in ADLS/Delta without losing the original change information.

```text
bronze/customer/
    batch_001/
    batch_002/
```

Include metadata such as:

```text
customer_id
name
operation
change_timestamp
batch_id
source_position
```

This gives you **auditability and replay capability**.

---

### 4. Process CDC in Databricks

Databricks reads the CDC data and applies the changes.

For example, using Delta Lake:

```sql
MERGE INTO target_customer AS target
USING source_customer AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND source.operation = 'DELETE'
    THEN DELETE

WHEN MATCHED AND source.operation = 'UPDATE'
    THEN UPDATE SET
        target.name = source.name

WHEN NOT MATCHED AND source.operation = 'INSERT'
    THEN INSERT (
        customer_id,
        name
    )
    VALUES (
        source.customer_id,
        source.name
    );
```

Conceptually:

```text
INSERT → Insert
UPDATE → Update
DELETE → Delete
```

---

## 5. Handle Duplicate CDC Events

A CDC stream can sometimes contain multiple changes for the same key within a batch.

For example:

```text
customer_id | operation | timestamp
------------|-----------|----------
101         | UPDATE    | 10:01
101         | UPDATE    | 10:05
101         | DELETE    | 10:10
```

Before merging, identify the **latest change** for each business key.

Using `ROW_NUMBER()`:

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY change_timestamp DESC
           ) AS rn
    FROM cdc_data
) x
WHERE rn = 1;
```

Then apply only the latest state.

---

## 6. Failure Handling

CDC pipelines need careful failure handling because losing the CDC position can result in **missing records**.

Recommended pattern:

```text
Read CDC Position
       ↓
Extract Changes
       ↓
Write Bronze
       ↓
Transform
       ↓
MERGE Target
       ↓
Validation
       ↓
SUCCESS?
   ↓       ↓
  YES      NO
   ↓       ↓
Update    Retry /
Position  Alert
```

### Critical rule:

**Don't advance the CDC watermark/LSN/SCN until the downstream target has successfully processed the changes.**

Otherwise:

```text
CDC Position Updated ❌
        ↓
Target Load Failed ❌
        ↓
Next Run Starts After Position
        ↓
Records Lost ❌
```

---

## 7. Initial Load + CDC

A common production design uses:

```text
Initial Full Load
       ↓
Capture CDC Position
       ↓
Start Incremental CDC
```

For example:

```text
Day 1:
10 Million records → Full Load

Day 2:
50,000 changes → CDC

Day 3:
25,000 changes → CDC
```

This is much more efficient than performing a 10-million-row full load every day.

---

## ⭐ Interview Answer

> **"For a CDC pipeline, I would first perform an initial full load and establish a CDC starting position such as an LSN, SCN, or source-specific change version. After that, I would use Azure Data Factory to orchestrate incremental extraction of inserts, updates, and deletes and land the CDC events into ADLS in a Bronze layer. Databricks would then process and deduplicate the changes and apply them to Delta tables using MERGE. I would maintain a control table containing the last successfully processed CDC position for each source table. I would update that position only after the target load and validation succeed. I would also implement retries, audit logging, data-quality checks, idempotent processing, and alerting so that the pipeline can safely recover from failures without losing or duplicating changes."**

### 🔥 Key CDC Design Points

Remember these **7 points** for an interview:

**Full Load → CDC Position → Extract Changes → Bronze → Deduplicate → MERGE → Update Position**

And for a **senior-level answer**, mention:

**Idempotency + Failure Recovery + Auditability + Replay + DELETE handling + Data Quality**.


11. Partition vs Bucketing

## Partitioning vs Bucketing in Spark

Both **partitioning** and **bucketing** improve query performance, but they solve different problems.

### 1. Partitioning

**Partitioning physically organizes data into separate directories based on a column's value.**

Example:

```python
df.write \
  .partitionBy("country") \
  .format("parquet") \
  .save("/data/customer")
```

Directory structure:

```text
customer/
├── country=India/
├── country=USA/
└── country=UK/
```

If you query:

```sql
SELECT *
FROM customer
WHERE country = 'India';
```

Spark can use **partition pruning** and read only:

```text
country=India/
```

instead of scanning the entire dataset.

### When to use partitioning?

Good partition columns are:

* Frequently used in filters
* Relatively low/medium cardinality
* Stable values

Examples:

```text
year
month
country
region
```

Avoid partitioning by very high-cardinality columns such as:

```text
customer_id
transaction_id
email
```

because it can create a huge number of small directories/files.

---

# 2. Bucketing

**Bucketing distributes rows into a fixed number of buckets based on the hash of one or more columns.**

Conceptually:

```text
customer_id
     ↓
Hash Function
     ↓
┌────┬────┬────┬────┐
│ B1 │ B2 │ B3 │ B4 │
└────┴────┴────┴────┘
```

Example:

```python
df.write \
  .bucketBy(10, "customer_id") \
  .sortBy("customer_id") \
  .saveAsTable("customer")
```

Rows with the same bucketing key are directed to the same bucket according to the bucketing scheme.

Bucketing can be useful for **repeated joins or aggregations on the bucket column**, because Spark can sometimes reduce shuffle work when the bucket layouts are compatible.

---

# Partitioning vs Bucketing

| Feature                      | Partitioning             | Bucketing                     |
| ---------------------------- | ------------------------ | ----------------------------- |
| Organization                 | Directories              | Fixed bucket files            |
| Based on                     | Column values            | Hash of column values         |
| Main benefit                 | Partition pruning        | Potentially reduce shuffle    |
| Best for                     | Filtering                | Joins / aggregations          |
| Cardinality                  | Prefer low/medium        | Can handle higher cardinality |
| Number of partitions/buckets | Based on distinct values | Explicit number of buckets    |
| Example                      | `country=India`          | `bucket 1, 2, 3...`           |

### Example

Suppose you have:

```text
1 Billion customer transactions
```

You frequently run:

```sql
WHERE year = 2026
```

Partition by:

```text
year
```

because Spark can skip other years.

But suppose you frequently join:

```sql
transactions JOIN customers
ON transactions.customer_id = customers.customer_id
```

Bucketing on `customer_id` can be useful when the tables are bucketed compatibly.

---

## ⭐ Interview Answer

> **"Partitioning and bucketing are both data-layout techniques, but they solve different problems. Partitioning physically separates data based on column values and is mainly used for partition pruning during filtering. For example, partitioning by year allows Spark to read only the required year's data. Bucketing distributes records into a fixed number of buckets based on a hash of the bucketing column and can help optimize repeated joins or aggregations on that column by reducing shuffle in compatible scenarios. I use partitioning for selective filters and bucketing when I have high-cardinality join or aggregation keys and a workload that benefits from a stable bucket layout."**

### 🔥 Easy Memory Trick

**Partitioning → "Which data should I READ?"**
**Bucketing → "How should related data be DISTRIBUTED?"**

**Interview tip:** Don't say *"bucketing always eliminates shuffle."* It **can** reduce shuffle only when Spark can exploit compatible bucket layouts and other query conditions are satisfied.


## 12. Handling Late-Arriving Data in Azure/Spark

**Late-arriving data** means data arrives **after its expected processing window**.

For example, an order belongs to **August 8**, but the source sends it on **August 10**.

```text
Expected:
Aug 8 → Order 101

Actually received:
Aug 10 → Order 101
```

This is common in batch pipelines, CDC systems, streaming, and data warehouses.

---

## 1. Separate Event Time and Processing Time

The first thing I would do is maintain both:

```text
event_time       → When the event actually happened
ingestion_time   → When we received the event
```

Example:

| Order | Event Time | Ingestion Time |
| ----- | ---------- | -------------- |
| 101   | Aug 8      | Aug 8          |
| 102   | Aug 8      | Aug 10         |
| 103   | Aug 10     | Aug 10         |

This allows us to identify late-arriving records.

```sql
WHERE ingestion_time > expected_processing_time
```

---

## 2. Use a Watermark

For incremental pipelines, maintain a **watermark/control value**.

But an important point is:

> **Don't use only ingestion time if records can arrive late.**

For example, if you process:

```text
Aug 10 data
```

and a record with `event_time = Aug 8` arrives on Aug 10, a simple event-time filter could miss it.

Instead, use a **lookback window**.

```text
Current watermark = Aug 10

Read data from:
Aug 8 → Aug 10
```

This gives the pipeline a chance to capture late records.

---

## 3. Lookback Window

A common production strategy is:

```text
Current Processing Date
        ↓
Look back 2-3 days
        ↓
Reprocess affected data
```

Example:

```text
Today's date = Aug 10

Process:
Aug 8
Aug 9
Aug 10
```

If an Aug 8 record arrives on Aug 10, it will still be picked up.

The exact lookback period depends on the source's expected lateness.

---

## 4. Use MERGE / Upsert

If you're using **Delta Lake**, use `MERGE` so that late-arriving records can update existing records instead of creating duplicates.

```sql
MERGE INTO target t
USING late_data s
ON t.order_id = s.order_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *
```

This makes the pipeline **idempotent**.

---

## 5. Handle Late-Arriving Dimensions

This is especially important in **Data Warehouse ETL**.

Suppose an order arrives:

```text
Order:
order_id = 1001
customer_id = 500
```

But Customer 500 hasn't arrived in the dimension table yet.

Instead of rejecting the order, you can create an **Unknown/Default dimension record**:

```text
customer_key = -1
customer_id  = UNKNOWN
```

Later, when the customer data arrives, you can update the fact record.

```text
Order
  ↓
Customer missing
  ↓
Use Unknown Key (-1)
  ↓
Customer arrives later
  ↓
Update dimension/fact relationship
```

---

## 6. Streaming: Watermarks

In Spark Structured Streaming, watermarks are commonly used to handle late events while controlling state size.

Conceptually:

```python
df.withWatermark("event_time", "2 hours")
```

This tells Spark that events can arrive up to approximately **2 hours late** for the relevant stateful processing.

For example:

```text
Current event time = 10:00

Allowed lateness = 2 hours

Events up to ~08:00 are considered within the allowed window.
```

The exact behavior depends on the query and windowing logic.

---

## 7. Reprocessing / Backfill

For significantly late data, I would support **backfill processing**.

Example:

```text
Normal pipeline:
Aug 10 → Aug 10 data

Late data:
Aug 7 data arrives on Aug 10

Backfill:
Aug 7 → Reprocess
```

A parameterized ADF pipeline can be useful:

```text
pipeline_start_date = 2026-08-07
pipeline_end_date   = 2026-08-10
```

---

## Production Design

```text
                 Source
                   |
                   v
             ADF / Streaming
                   |
                   v
             Bronze Layer
                   |
                   v
       Event Time + Ingestion Time
                   |
                   v
          Late Data Detection
             /           \
            /             \
      On-Time             Late
         |                   |
         v                   v
    Normal Flow        Lookback/Replay
         |                   |
         └─────────┬─────────┘
                   v
              Delta MERGE
                   |
                   v
              Silver/Gold
```

---

## ⭐ Interview Answer

> **"For late-arriving data, I first distinguish between event time and ingestion time so that I know when the record actually occurred versus when it reached the pipeline. For batch processing, I generally use a lookback window rather than relying strictly on the latest watermark. For example, instead of processing only today's data, I may reprocess the previous two or three days depending on the source SLA. I make the target load idempotent using Delta MERGE or an equivalent upsert strategy so reprocessing doesn't create duplicates. For late-arriving dimension data, I can use an unknown or default dimension key and resolve it when the dimension record arrives. For streaming workloads, I use Spark watermarks to control how long late events are accepted and how much state is retained. For data that arrives beyond the normal lateness threshold, I support a parameterized backfill or replay process."**

### 🔥 Key Points to Remember

**Event Time → Ingestion Time → Lookback → MERGE → Watermark → Backfill**

And the most important interview statement:

> **"I don't blindly advance the watermark based only on processing time; I design the pipeline to tolerate expected lateness and make reprocessing idempotent."**
